# Módulo 14 · Práctica: anatomía de un modelo de 7.000 millones

Vas a calcular **a mano** los parámetros, los FLOPs y la memoria de LLaMA-2 7B a partir de su configuración pública, y después lo comprobarás **con código**: construirás un Transformer decoder-only estilo LLaMA en PyTorch, lo instanciarás en versión diminuta y en tamaño real (sin reservar memoria) y contarás sus parámetros.

| Parte | Qué haces |
| --- | --- |
| 1 | Contar parámetros con fórmulas |
| 2 | Memoria de pesos, gradientes y Adam |
| 3 | Piezas del bloque: RMSNorm, RoPE y SwiGLU |
| 4 | El modelo completo y el recuento con código |
| 5 | LLaMA-2 7B real en el dispositivo `meta` |
| 6 | FLOPs: comprobar 2N y 6N con un contador |
| 7 | Retos |

Las celdas `# TU CÓDIGO` son ejercicios; la celda `# Comprobación` siguiente te dice si están bien.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

# Configuración pública de LLaMA-2 7B (Touvron et al., 2023)
LLAMA2_7B = dict(d=4096, L=32, h=32, d_ff=11008, V=32000, tied=False)

## Parte 1 · Contar parámetros con fórmulas

Un bloque LLaMA no tiene sesgos. Contiene $W_Q, W_K, W_V, W_O$ ($d\times d$ cada una), un MLP SwiGLU con tres matrices ($d\times d_{ff}$, $d\times d_{ff}$, $d_{ff}\times d$) y dos RMSNorm con $d$ pesos cada una. Fuera de los bloques: embedding $V\times d$, RMSNorm final ($d$) y unembedding $d\times V$ (salvo que esté atado).

$$N = L\,(4d^2 + 3d\,d_{ff} + 2d) + Vd\,(1 + [\text{no atado}]) + d$$

**Ejercicio 1.** Implementa `params_bloque(cfg)` y `params_total(cfg)`.

In [ ]:
def params_bloque(cfg):
    d, d_ff = cfg["d"], cfg["d_ff"]
    # TU CÓDIGO
    return None

def params_total(cfg):
    # TU CÓDIGO (usa params_bloque y cfg["tied"])
    return None

print(params_bloque(LLAMA2_7B), params_total(LLAMA2_7B))

In [ ]:
# Comprobación
assert params_bloque(LLAMA2_7B) == 202_383_360, "Bloque: 4d² + 3·d·d_ff + 2d"
assert params_total(LLAMA2_7B) == 6_738_415_616, "Total: L·bloque + 2·V·d + d"
assert params_total({**LLAMA2_7B, "tied": True}) == 6_738_415_616 - 32000 * 4096
print("Correcto: LLaMA-2 7B tiene", f"{params_total(LLAMA2_7B):,}".replace(",", "."), "parámetros")

**Ejercicio 2.** Calcula la aproximación $12Ld^2$ y la fracción del total que suponen el MLP (`frac_mlp`) y los embeddings (`frac_emb`).

In [ ]:
# TU CÓDIGO
aprox = None
frac_mlp = None
frac_emb = None

In [ ]:
# Comprobación
assert aprox == 6_442_450_944
assert abs(frac_mlp - 0.6424) < 1e-3, "El MLP es casi 2/3 del modelo"
assert abs(frac_emb - 0.0389) < 1e-3
print(f"12Ld² = {aprox/1e9:.3f}e9 ({aprox/params_total(LLAMA2_7B):.1%} del total); MLP {frac_mlp:.1%}; embeddings {frac_emb:.1%}")

## Parte 2 · Memoria

| Uso | Bytes por parámetro |
| --- | --- |
| Inferencia bf16 | 2 |
| Inferencia int4 | 0,5 |
| Entrenar con Adam en precisión mixta: pesos bf16 + gradientes bf16 + copia fp32 + m + v | 2 + 2 + 4 + 4 + 4 = 16 |

**Ejercicio 3.** Escribe `memoria_gb(N, bytes_por_param)` (GB = 10⁹ bytes) y calcula la memoria de inferencia en bf16 y la de entrenamiento de LLaMA-2 7B. Calcula también `N_max_24`, el mayor número de parámetros que se puede entrenar así en una GPU de 24 GB (sin activaciones).

In [ ]:
# TU CÓDIGO
def memoria_gb(N, bytes_por_param):
    return None

mem_inf = None
mem_train = None
N_max_24 = None

In [ ]:
# Comprobación
assert abs(mem_inf - 13.48) < 0.01, "bf16 = 2 bytes por parámetro"
assert abs(mem_train - 107.81) < 0.01, "Adam mixto = 16 bytes por parámetro"
assert abs(N_max_24 - 1.5e9) < 1e6
print(f"Inferencia: {mem_inf:.1f} GB · Entrenamiento: {mem_train:.1f} GB · En 24 GB se entrenan {N_max_24/1e9:.1f}e9 parámetros")

## Parte 3 · Piezas del bloque

RMSNorm ya la conoces del módulo 11. Aquí implementas RoPE y SwiGLU.

**RoPE.** Para una cabeza de dimensión $d_h$, agrupa las componentes en pares $(x_{2i}, x_{2i+1})$ y gira el par $i$ un ángulo $m\,\theta_i$, con $\theta_i = 10000^{-2i/d_h}$ y $m$ la posición:

$$\begin{pmatrix}x'_{2i}\\ x'_{2i+1}\end{pmatrix} = \begin{pmatrix}\cos m\theta_i & -\sin m\theta_i\\ \sin m\theta_i & \cos m\theta_i\end{pmatrix}\begin{pmatrix}x_{2i}\\ x_{2i+1}\end{pmatrix}$$

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

def angulos_rope(n, d_h, base=10000.0):
    theta = base ** (-torch.arange(0, d_h, 2).float() / d_h)   # (d_h/2,)
    m = torch.arange(n).float()                                 # (n,)
    return torch.outer(m, theta)                                # (n, d_h/2): ángulo m·θ_i

**Ejercicio 4.** Completa `aplicar_rope(x, ang)`. `x` tiene forma `(..., n, d_h)` y `ang` forma `(n, d_h/2)`. Pista: `x[..., 0::2]` son las componentes pares.

In [ ]:
def aplicar_rope(x, ang):
    c, s = torch.cos(ang), torch.sin(ang)
    x0, x1 = x[..., 0::2], x[..., 1::2]
    # TU CÓDIGO: calcula y0, y1 (la rotación) y vuelve a intercalarlos
    y0, y1 = x0, x1
    return torch.stack([y0, y1], dim=-1).flatten(-2)

In [ ]:
# Comprobación 1: el ejemplo del documento (d_h = 2, θ = π/6, q = (1, 2), k = (3, 1))
th = math.pi / 6
q = torch.tensor([[1., 2.]]); k = torch.tensor([[3., 1.]])
ang = lambda m: torch.tensor([[m * th]])
s = (aplicar_rope(q, ang(3)) * aplicar_rope(k, ang(1))).sum().item()
assert abs(s - (-1.8301)) < 1e-3, f"Esperaba -1,830 y sale {s:.4f}"
# Comprobación 2: la puntuación solo depende de n - m
d_h, n = 16, 12
q = torch.randn(d_h); k = torch.randn(d_h)
A = angulos_rope(n, d_h)
Q = aplicar_rope(q.expand(n, d_h), A); K = aplicar_rope(k.expand(n, d_h), A)
S = Q @ K.T                                    # S[m, n] = q̃_m · k̃_n
assert torch.allclose(S[5, 2], S[9, 6], atol=1e-4) and torch.allclose(S[3, 3], S[0, 0], atol=1e-4)
assert torch.allclose(Q.norm(dim=-1), q.norm().expand(n), atol=1e-4), "La rotación conserva la norma"
print("Correcto. S[5,2] =", round(S[5, 2].item(), 4), "= S[9,6] =", round(S[9, 6].item(), 4))

**Ejercicio 5.** Completa el MLP SwiGLU: $(\mathrm{SiLU}(xW_g)\odot xW_u)W_d$, sin sesgos.

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, d, d_ff):
        super().__init__()
        # TU CÓDIGO: tres nn.Linear sin sesgo: gate, up, down
        pass
    def forward(self, x):
        # TU CÓDIGO
        return x

In [ ]:
# Comprobación
mlp = SwiGLU(8, 20)
assert sum(p.numel() for p in mlp.parameters()) == 3 * 8 * 20, "Deben ser 3·d·d_ff parámetros"
x = torch.randn(3, 8)
ref = mlp.down(F.silu(mlp.gate(x)) * mlp.up(x))
assert torch.allclose(mlp(x), ref)
print("Correcto")

## Parte 4 · El modelo completo

El bloque es pre-norm: `x = x + attn(norm1(x))` y `x = x + mlp(norm2(x))`. La atención es la del módulo 13: puntuaciones escaladas, máscara causal y softmax.

In [ ]:
class Atencion(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.wq = nn.Linear(d, d, bias=False); self.wk = nn.Linear(d, d, bias=False)
        self.wv = nn.Linear(d, d, bias=False); self.wo = nn.Linear(d, d, bias=False)
    def forward(self, x, ang):
        B, n, d = x.shape
        sep = lambda t: t.view(B, n, self.h, self.d_h).transpose(1, 2)   # (B, h, n, d_h)
        q, k, v = sep(self.wq(x)), sep(self.wk(x)), sep(self.wv(x))
        q, k = aplicar_rope(q, ang), aplicar_rope(k, ang)                 # RoPE solo en q y k
        S = q @ k.transpose(-2, -1) / math.sqrt(self.d_h)               # (B, h, n, n)
        mascara = torch.ones(n, n, dtype=torch.bool, device=x.device).triu(1)
        A = torch.softmax(S.masked_fill(mascara, float("-inf")), dim=-1)  # causal (módulo 13)
        y = A @ v
        return self.wo(y.transpose(1, 2).reshape(B, n, d))

class Bloque(nn.Module):
    def __init__(self, d, h, d_ff):
        super().__init__()
        self.norm1, self.attn = RMSNorm(d), Atencion(d, h)
        self.norm2, self.mlp = RMSNorm(d), SwiGLU(d, d_ff)
    def forward(self, x, ang):
        x = x + self.attn(self.norm1(x), ang)   # pre-norm + residual
        x = x + self.mlp(self.norm2(x))
        return x

class MiniLlama(nn.Module):
    def __init__(self, d, L, h, d_ff, V, tied=False):
        super().__init__()
        self.d_h = d // h
        self.tok_emb = nn.Embedding(V, d)
        self.bloques = nn.ModuleList([Bloque(d, h, d_ff) for _ in range(L)])
        self.norm_f = RMSNorm(d)
        self.lm_head = nn.Linear(d, V, bias=False)
        if tied:
            self.lm_head.weight = self.tok_emb.weight   # weight tying
    def forward(self, idx):
        x = self.tok_emb(idx)
        ang = angulos_rope(idx.shape[1], self.d_h).to(x.device)
        for b in self.bloques:
            x = b(x, ang)
        return self.lm_head(self.norm_f(x))

def contar(model):
    return sum(p.numel() for p in model.parameters())   # los tensores atados cuentan una vez

**Ejercicio 6.** Crea una configuración diminuta `TINY` con $d = 64$, $L = 2$, $h = 4$, vocabulario 100 y $d_{ff}$ = el múltiplo de 4 inmediatamente superior a $8d/3$. Instancia el modelo y cuenta sus parámetros con `contar`.

In [ ]:
# TU CÓDIGO
TINY = None      # dict(d=..., L=..., h=..., d_ff=..., V=..., tied=False)
modelo = None
n_codigo = None

In [ ]:
# Comprobación
assert TINY["d_ff"] == 172, "8·64/3 = 170,67 → 172"
assert n_codigo == params_total(TINY), f"Código {n_codigo} ≠ fórmula {params_total(TINY)}"
logits = modelo(torch.randint(0, 100, (2, 10)))
assert logits.shape == (2, 10, 100)
print("Correcto: fórmula y código coinciden en", n_codigo, "parámetros")
for nombre, p in list(modelo.named_parameters())[:12]:
    print(f"{nombre:28s} {tuple(p.shape)}")

**Pregunta.** Comprueba que la causalidad funciona: cambia el último token de la entrada y verifica que los logits de las posiciones anteriores no cambian.

In [ ]:
idx = torch.randint(0, 100, (1, 8))
idx2 = idx.clone(); idx2[0, -1] = (idx[0, -1] + 1) % 100
with torch.no_grad():
    print("¿Posiciones 0..6 iguales?", torch.allclose(modelo(idx)[:, :-1], modelo(idx2)[:, :-1], atol=1e-5))

**Ejercicio 7.** Instancia la versión atada (`tied=True`) y guarda en `ahorro` cuántos parámetros menos tiene. Compara con $V\cdot d$.

In [ ]:
# TU CÓDIGO
modelo_atado = None
ahorro = None

In [ ]:
# Comprobación
assert ahorro == TINY["V"] * TINY["d"], "Atar ahorra exactamente V·d parámetros"
assert modelo_atado.lm_head.weight is modelo_atado.tok_emb.weight
print(f"Correcto: ahorro de {ahorro} parámetros ({ahorro / contar(modelo):.1%} del modelo diminuto)")

## Parte 5 · LLaMA-2 7B de verdad, sin memoria

En el dispositivo `meta`, PyTorch crea los tensores con su forma pero **sin datos**. Así puedes «instanciar» 6.700 millones de parámetros en un portátil y contarlos.

In [ ]:
# TU CÓDIGO: instancia MiniLlama con LLAMA2_7B dentro de `with torch.device("meta"):` y cuenta
n_7b = None

In [ ]:
# Comprobación
assert n_7b == 6_738_415_616, "Debe coincidir con la cifra oficial de LLaMA-2 7B"
assert n_7b == params_total(LLAMA2_7B)
print("Correcto: el código cuenta", f"{n_7b:,}".replace(",", "."), "parámetros")

## Parte 6 · FLOPs: comprobar 2N y 6N

`FlopCounterMode` cuenta los FLOPs de cada multiplicación matricial (2·m·n·k). El embedding es una consulta en tabla, sin FLOPs, así que el número relevante es $N_{mat} = N - Vd - (\text{normas})$. Con $T$ tokens esperamos:

$$\text{forward} \approx 2\,N_{mat}\,T + \underbrace{4\,L\,T^2 d}_{QK^\top \text{ y } AV}, \qquad \text{backward} \approx 2 \times \text{forward}$$

(Nuestra atención calcula la matriz $T\times T$ completa aunque la máscara anule la mitad.)

In [ ]:
from torch.utils.flop_counter import FlopCounterMode

def medir_flops(model, T, backward=False):
    idx = torch.randint(0, TINY["V"], (1, T))
    with FlopCounterMode(display=False) as fc:
        out = model(idx)
        if backward:
            out.sum().backward()
    return fc.get_total_flops()

T = 32
d, L, V = TINY["d"], TINY["L"], TINY["V"]
N_mat = sum(p.numel() for n_, p in modelo.named_parameters() if p.dim() == 2) - V * d
f_fwd = medir_flops(modelo, T)
f_tot = medir_flops(modelo, T, backward=True)
print("FLOPs forward medidos:", f_fwd, "· total fwd+bwd:", f_tot)

**Ejercicio 8.** Calcula la predicción `pred_fwd` = $2N_{mat}T + 4LT^2d$ y el cociente `ratio` = total/forward medidos.

In [ ]:
# TU CÓDIGO
pred_fwd = None
ratio = None

In [ ]:
# Comprobación
assert pred_fwd == f_fwd, f"Predicción {pred_fwd} ≠ medida {f_fwd}"
assert abs(ratio - 3) < 0.05, "Forward + backward ≈ 3 × forward  →  6N por token"
print(f"Correcto: forward exacto; entrenar cuesta {ratio:.2f} veces el forward")

**Ejercicio 9.** Para LLaMA-2 7B calcula: `flops_token` (≈ 2N), `C` (≈ 6ND con $D = 2\cdot10^{12}$) y `horas_gpu` en una A100 (312·10¹² FLOP/s) al 40 % de aprovechamiento.

In [ ]:
# TU CÓDIGO
flops_token = None
C = None
horas_gpu = None

In [ ]:
# Comprobación
assert abs(flops_token - 1.348e10) / 1.348e10 < 1e-3
assert abs(C - 8.086e22) / 8.086e22 < 1e-3
assert abs(horas_gpu - 179_979) < 10
print(f"2N = {flops_token:.3e} FLOPs/token · C = {C:.3e} FLOPs · {horas_gpu:,.0f} h de A100 (LLaMA-2 declara 184.320)")

## Parte 7 · Retos

**Reto A.** Usa tus funciones para LLaMA-2 13B (d = 5120, L = 40, h = 40, d_ff = 13.824, V = 32.000) y comprueba el resultado instanciándolo en `meta`.

**Reto B.** Añade a `Bloque` un parámetro `post_norm=True` que implemente `x = norm1(x + attn(x))`. ¿Cambia el número de parámetros? ¿Qué pieza deja de ser necesaria?

**Reto C.** Con $n$ tokens de contexto la atención añade $4Lnd$ FLOPs por token. Calcula para LLaMA-2 7B la longitud de contexto en que iguala a $2N$.

In [ ]:
# TU CÓDIGO (reto A y C)
LLAMA2_13B = None
n_13b = None
n_igual = None

In [ ]:
# Comprobación
assert n_13b == params_total(LLAMA2_13B) == 13_015_864_320
assert abs(n_igual - 25_705) < 1
print(f"Correcto: 13B = {n_13b:,} parámetros; la atención iguala a 2N con {n_igual:,.0f} tokens".replace(",", "."))

## Soluciones de referencia

Intenta los ejercicios antes de mirar. Para usar una solución, copia y descomenta.

In [ ]:
# # ---- Ejercicio 1 ----
# def params_bloque(cfg):
#     d, d_ff = cfg["d"], cfg["d_ff"]
#     return 4 * d * d + 3 * d * d_ff + 2 * d
#
# def params_total(cfg):
#     d, L, V = cfg["d"], cfg["L"], cfg["V"]
#     emb = V * d * (1 if cfg["tied"] else 2)
#     return L * params_bloque(cfg) + emb + d
#
# print(params_bloque(LLAMA2_7B), params_total(LLAMA2_7B))
#
# # ---- Ejercicio 2 ----
# d, L, d_ff, V = 4096, 32, 11008, 32000
# N = params_total(LLAMA2_7B)
# aprox = 12 * L * d**2
# frac_mlp = L * 3 * d * d_ff / N
# frac_emb = 2 * V * d / N
#
# # ---- Ejercicio 3 ----
# def memoria_gb(N, bytes_por_param):
#     return N * bytes_por_param / 1e9
#
# N = params_total(LLAMA2_7B)
# mem_inf = memoria_gb(N, 2)
# mem_train = memoria_gb(N, 2 + 2 + 4 + 4 + 4)
# N_max_24 = 24e9 / 16
#
# # ---- Ejercicio 4 ----
# def aplicar_rope(x, ang):
#     c, s = torch.cos(ang), torch.sin(ang)
#     x0, x1 = x[..., 0::2], x[..., 1::2]
#     y0 = x0 * c - x1 * s
#     y1 = x0 * s + x1 * c
#     return torch.stack([y0, y1], dim=-1).flatten(-2)
#
# # ---- Ejercicio 5 ----
# class SwiGLU(nn.Module):
#     def __init__(self, d, d_ff):
#         super().__init__()
#         self.gate = nn.Linear(d, d_ff, bias=False)
#         self.up = nn.Linear(d, d_ff, bias=False)
#         self.down = nn.Linear(d_ff, d, bias=False)
#     def forward(self, x):
#         return self.down(F.silu(self.gate(x)) * self.up(x))
#
# # ---- Ejercicio 6 ----
# TINY = dict(d=64, L=2, h=4, d_ff=4 * math.ceil(8 * 64 / 3 / 4), V=100, tied=False)
# modelo = MiniLlama(**TINY)
# n_codigo = contar(modelo)
#
# # ---- Ejercicio 7 ----
# modelo_atado = MiniLlama(**{**TINY, "tied": True})
# ahorro = contar(modelo) - contar(modelo_atado)
#
# # ---- Ejercicio 8 ----
# with torch.device("meta"):
#     m7 = MiniLlama(**LLAMA2_7B)
# n_7b = contar(m7)
#
# # ---- Ejercicio 9 ----
# pred_fwd = 2 * N_mat * T + 4 * L * T**2 * d
# ratio = f_tot / f_fwd
#
# # ---- Ejercicio retos ----
# N = params_total(LLAMA2_7B)
# flops_token = 2 * N
# C = 6 * N * 2e12
# horas_gpu = C / (0.4 * 312e12) / 3600
#
# # ---- Ejercicio retos ----
# LLAMA2_13B = dict(d=5120, L=40, h=40, d_ff=13824, V=32000, tied=False)
# with torch.device("meta"):
#     n_13b = contar(MiniLlama(**LLAMA2_13B))
# N = params_total(LLAMA2_7B)
# n_igual = N / (2 * 32 * 4096)